# Assignment 4: финальный эксперимент (Kazakh QA: original vs translated)

**Перед запуском:** загрузите форму Protocol Freeze (Appendix A) в Moodle. Время загрузки должно быть *раньше* времени этого запуска.

Runtime → Change runtime type → **T4 GPU**. Затем Runtime → Run all. Примерно 20–30 минут.

Ничего не меняйте в настройках: они заморожены в `configs/protocol.json`.

In [ ]:
REPO_URL = "https://github.com/<ваш-аккаунт>/kazakh-qa-origin.git"   # <-- вставьте ссылку на ваш репозиторий
!git clone -q $REPO_URL repo
%cd repo
!git log -1 --format="%H %cd"

In [ ]:
%pip -q install -r requirements.txt

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!python qa_compare.py self-test
!sha256sum results_main/selected.json   # должно быть 049831f6...

## 1. Простой baseline (CPU)

In [ ]:
!python a4_experiment.py lexical

## 2. Сильный baseline из литературы: XLM-R-large, дообученный на SQuAD 2.0

In [ ]:
!python a4_experiment.py xlmr

## 3. Основное условие: Qwen2.5-1.5B-Instruct с пассажем

In [ ]:
!python qa_compare.py run --out results_main --revision 989aa7980e4cf806f80c7fef2b1adb7bc71aa306
!python a4_experiment.py import-qwen

## 4. Абляция: Qwen без пассажа (closed-book)

In [ ]:
!python a4_experiment.py qwen-closed

## 5. Анализ: таблицы, тесты, графики, лист для разбора ошибок

In [ ]:
!python a4_experiment.py analyse
from IPython.display import Image, display
import pandas as pd
display(pd.read_csv("results_main/analysis/main_results.csv"))
for f in ["fig1_scores_by_condition", "fig2_origin_gap_forest", "fig3_subgroup_answer_type"]:
    display(Image(f"results_main/analysis/{f}.png"))

## 6. Скачайте архив и отправьте его (результаты + логи)

In [ ]:
!pip freeze > results_main/pip_freeze_colab.txt
import shutil
shutil.make_archive("/content/a4_results", "zip", "results_main")
from google.colab import files
files.download("/content/a4_results.zip")